In [ ]:
# Auto-injected: export every displayed figure as PNG + PDF + SVG into png/ pdf/ svg/
import os
import matplotlib.pyplot as plt
_PDF_STEM = 'fig3_3'
_PDF_N = [0]
for _d in ('pdf', 'svg', 'png'):
    os.makedirs(_d, exist_ok=True)
def _save_fig(base, fig=None, dpi=300, **kw):
    f = fig if fig is not None else plt.gcf()
    f.savefig(f'pdf/{base}.pdf', bbox_inches='tight', **kw)
    f.savefig(f'svg/{base}.svg', bbox_inches='tight', **kw)
    f.savefig(f'png/{base}.png', dpi=dpi, bbox_inches='tight', **kw)
def _save_pdf():
    _PDF_N[0] += 1
    _save_fig(f'{_PDF_STEM}_{_PDF_N[0]:02d}')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
import matplotlib

# 设置全局字体为Arial
plt.rcParams.update({
    'font.family': 'Arial',  # 设置字体为Arial
    'font.size': 16,  # 基础字体大小
})

# =============================================================================
# 数据预处理部分 - 三个情景的四列桑基图数据准备
# =============================================================================

# 1. 读取岛屿可行性分析数据
data_path = Path("../result/island_viability_summary_electric.csv")
df = pd.read_csv(data_path)  # 加载原始数据
print(f"原始数据形状: {df.shape}")  # 显示数据维度信息

# 2. 加载IPCC区域地理数据用于区域分配
try:
    ipcc_regions = gpd.read_file("IPCC-WGI-reference-regions-v4.geojson")  # 加载IPCC区域边界数据
    ipcc_regions.sindex  # 创建空间索引以加速地理查询
    print("IPCC区域地理数据加载成功")
except Exception as e:
    print(f"无法加载IPCC区域数据: {e}")
    ipcc_regions = None

# 3. 定义IPCC区域分配函数（高效版本）
def assign_ipcc_region_optimized(lat, lon, ipcc_regions_gdf):
    """使用空间索引高效地将坐标分配到IPCC区域"""
    point = Point(lon, lat)  # 创建点几何对象
    possible_matches_index = list(ipcc_regions_gdf.sindex.intersection(point.bounds))  # 空间索引快速筛选
    possible_matches = ipcc_regions_gdf.iloc[possible_matches_index]  # 获取可能匹配的区域
    precise_matches = possible_matches[possible_matches.contains(point)]  # 精确几何匹配
    if not precise_matches.empty:
        return precise_matches.iloc[0]['Acronym']  # 返回区域简称
    return 'Unknown'  # 未匹配到区域时返回Unknown

# 4. 定义变化计算函数（与上面cell保持一致）
def calculate_position_change(df_base, df_compare):
    """计算两个情景之间的位置变化 - 修正版，包含IPCC区域信息"""
    # 修正：合并时包含ipcc_region信息
    merged = pd.merge(df_base[['island_id', 'tariff_breakeven', 'ipcc_region']],
                      df_compare[['island_id', 'tariff_breakeven']],
                      on='island_id', suffixes=('_base', '_compare'))
    merged['position_change'] = abs(merged['tariff_breakeven_compare'] - merged['tariff_breakeven_base'])
    # 重命名ipcc_region列以保持一致性
    merged = merged.rename(columns={'ipcc_region': 'ipcc_region_base'})
    return merged

# 5. 为整个数据集分配IPCC区域（与上面cell保持一致）
if ipcc_regions is not None:
    print("正在为岛屿分配IPCC区域...")
    df['ipcc_region'] = df.apply(
        lambda row: assign_ipcc_region_optimized(row['lat'], row['lon'], ipcc_regions), axis=1
    )  # 应用区域分配函数
    print(f"区域分配完成，涉及区域: {df['ipcc_region'].nunique()}")
else:
    df['ipcc_region'] = 'Unknown'  # 如果没有区域数据，设为Unknown

df_filtered = df
# =============================================================================
# 6. 在有显著变化的岛屿中，再筛选有效区域（岛屿数量阈值）
# =============================================================================

print("\n=== 在有显著变化的岛屿中应用区域数量阈值 ===")

MIN_ISLANDS_PER_REGION = 10  # 每个区域最少岛屿数量阈值

# 统计有显著变化的岛屿在各区域的分布
region_counts_filtered = df_filtered['ipcc_region'].value_counts()  # 统计各区域岛屿数量
valid_regions = region_counts_filtered[region_counts_filtered > MIN_ISLANDS_PER_REGION].index.tolist()  # 筛选有效区域

print(f"有显著变化岛屿的区域分布:")
print(region_counts_filtered.head(10))
print(f"满足数量阈值的有效区域 (岛屿数量>{MIN_ISLANDS_PER_REGION}): {len(valid_regions)} 个")
print(f"有效区域列表: {valid_regions}")

# 最终筛选：既有显著变化又在有效区域的岛屿
df_final = df_filtered[df_filtered['ipcc_region'].isin(valid_regions)].copy()
final_island_ids = df_final['island_id'].unique()

print(f"最终用于桑基图的岛屿数: {len(final_island_ids)}")
print(f"最终数据形状: {df_final.shape}")

# <<< MODIFIED 1: Define the six region names >>>
# --- 六区域定义 ---
SIX_REGION_NAMES = [
    "Feasible\nLow Cost\nHigh Affordability",      # 区域1
    "Feasible\nHigh Cost\nHigh Affordability",     # 区域2
    "Feasible\nLow Cost\nLow Affordability",       # 区域3
    "Infeasible\nHigh Cost\nHigh Affordability",   # 区域4
    "Infeasible\nLow Cost\nLow Affordability",     # 区域5
    "Infeasible\nHigh Cost\nLow Affordability",    # 区域6
]

# <<< MODIFIED 2: Replace the classification function >>>
# --- 核心分类函数 (六区域模型) ---
def classify_six_regions(row, median_breakeven):
    """根据成本中位数和可行性对单个点进行六区域分类"""
    x = row['tariff_breakeven']
    y = row['tariff_affordable']
    
    is_low_cost = x <= median_breakeven
    is_high_affordability_benchmark = y > median_breakeven
    is_feasible = y >= x
    
    if is_feasible:
        if is_low_cost and is_high_affordability_benchmark: return SIX_REGION_NAMES[0]
        elif not is_low_cost and is_high_affordability_benchmark: return SIX_REGION_NAMES[1]
        elif is_low_cost and not is_high_affordability_benchmark: return SIX_REGION_NAMES[2]
        else: return SIX_REGION_NAMES[5]  # Feasible, High Cost, Low Affordability
    else: # Infeasible
        if not is_low_cost and is_high_affordability_benchmark: return SIX_REGION_NAMES[3]
        elif is_low_cost and not is_high_affordability_benchmark: return SIX_REGION_NAMES[4]
        else: return SIX_REGION_NAMES[5] # Infeasible, High Cost, Low Affordability

# =============================================================================
# 8. 处理三个情景的数据并计算统一的中位数分类基准（修正版）
# =============================================================================

scenarios = ['output_0', 'output_2020', 'output_2050']  # 三个主要情景
scenario_labels = ['Ideal', 'Baseline', 'Climate Stress']  # 情景标签

# 准备合并数据的列表
combined_data = []

# <<< MODIFIED 3: Use a single median from output_0 as the global benchmark >>>
# 首先从output_0（Ideal情景）计算全局中位数基准
df_output_0 = df_final[df_final['scenario'] == 'output_0'].copy()  # 获取Ideal情景数据
global_median_breakeven = df_output_0['tariff_breakeven'].median()  # 全局盈亏平衡电价中位数

print(f"=== 使用统一的output_0中位数基准进行所有情景的六区域分类 ===")
print(f"全局基准 - 盈亏平衡电价中位数: {global_median_breakeven:.3f}")

# 为每个情景使用统一的全局中位数进行分类（只针对最终筛选的岛屿）
for scenario in scenarios:
    df_scenario = df_final[df_final['scenario'] == scenario].copy()  # 筛选情景数据

    # <<< MODIFIED 4: Call the new classification function and name the new column 'classification' >>>
    # 使用统一的全局中位数基准进行六区域分类
    df_scenario['classification'] = df_scenario.apply(
        lambda row: classify_six_regions(row, global_median_breakeven), axis=1
    )  # 为每个岛屿分配区域

    print(f"情景 {scenario}: 有效岛屿数 {len(df_scenario)}, 使用统一基准分类")

    combined_data.append(df_scenario)

# =============================================================================
# 9. 创建四列桑基图的数据结构
# =============================================================================

def prepare_four_column_sankey_data(df_list):
    """为四列桑基图准备数据"""

    # 找到所有情景中共同存在的岛屿
    common_islands = set(df_list[0]['island_id'])
    for df_scenario in df_list[1:]:
        common_islands = common_islands.intersection(set(df_scenario['island_id']))

    print(f"三个情景共同存在的岛屿数量: {len(common_islands)}")

    # 为共同岛屿创建桑基图数据
    sankey_data = []

    for island_id in common_islands:
        # 获取该岛屿在三个情景下的信息
        island_info = {}
        for i, df_scenario in enumerate(df_list):
            island_row = df_scenario[df_scenario['island_id'] == island_id].iloc[0]
            # <<< MODIFIED 5: Use the new 'classification' column >>>
            island_info[f'scenario_{i}'] = {
                'region': island_row['ipcc_region'],
                'classification': island_row['classification'] # Changed from 'quadrant'
            }

        # 添加到桑基图数据
        sankey_data.append({
            'island_id': island_id,
            'region': island_info['scenario_0']['region'],              # 第一列：区域
            'ideal_class': island_info['scenario_0']['classification'], # 第二列：Ideal分类
            'baseline_class': island_info['scenario_1']['classification'],# 第三列：Baseline分类
            'climate_class': island_info['scenario_2']['classification'] # 第四列：Climate Stress分类
        })

    return pd.DataFrame(sankey_data)

# =============================================================================
# 10. 准备桑基图数据
# =============================================================================

sankey_df = prepare_four_column_sankey_data(combined_data)
print(f"\n=== 最终桑基图数据摘要 ===")
print(f"桑基图数据准备完成，包含 {len(sankey_df)} 个岛屿")
print(f"涉及区域: {sankey_df['region'].nunique()} 个")
print(f"区域列表: {sorted(sankey_df['region'].unique())}")

# 打印筛选过程摘要
print(f"\n=== 筛选过程摘要 ===")
print(f"1. 满足区域数量阈值的岛屿: {len(final_island_ids)}")
print(f"2. 最终桑基图岛屿数 (在三个情景中都存在): {len(sankey_df)}")
print(f"3. 最终涉及区域数: {len(valid_regions)}")

In [ ]:
sankey_df

In [ ]:
# 1. 识别发生过分类变化的岛屿
changed_island_ids = []
for _, row in sankey_df.iterrows():
    # 如果三个情景的分类不完全相同，就认为发生了变化
    if not (row['ideal_class'] == row['baseline_class'] == row['climate_class']):
        changed_island_ids.append(row['island_id'])

print(f"发生过分类变化的岛屿数量: {len(changed_island_ids)}")

# 2. 获取这些变化岛屿所在的区域
changed_regions = sankey_df[sankey_df['island_id'].isin(changed_island_ids)]['region'].unique()
print(f"有岛屿发生变化的区域: {sorted(changed_regions)}")

# 3. 在有岛屿发生变化的区域中，筛选岛屿数量大于10个的区域
MIN_ISLANDS_PER_REGION = 10  # 每个区域最少岛屿数量阈值
region_counts_in_changed = sankey_df[sankey_df['region'].isin(changed_regions)]['region'].value_counts()
valid_changed_regions = region_counts_in_changed[region_counts_in_changed >= MIN_ISLANDS_PER_REGION].index.tolist()

print(f"有变化区域的岛屿数量分布:")
for region in sorted(changed_regions):
    count = region_counts_in_changed.get(region, 0)
    status = "✓" if count > MIN_ISLANDS_PER_REGION else "✗"
    print(f"  {region}: {count} 个岛屿 {status}")

print(f"满足数量阈值(>{MIN_ISLANDS_PER_REGION})的有效变化区域: {len(valid_changed_regions)} 个")
print(f"有效区域列表: {sorted(valid_changed_regions)}")

# 4. 筛选出这些有效区域内的所有岛屿（包括未变化的）
sankey_df_filtered = sankey_df[sankey_df['region'].isin(valid_changed_regions)].copy()

print(f"原始数据: {len(sankey_df)} 个岛屿")
print(f"筛选后数据: {len(sankey_df_filtered)} 个岛屿")

# 现在sankey_df_filtered就是你要的结果
sankey_df_filtered

In [ ]:
# 只保留实际发生分类变化的岛屿，而不是整个区域
# 这样桑基图会更精准地反映真正发生变化的情况

# 1. 识别发生过分类变化的岛屿
changed_island_ids = []
for _, row in sankey_df_filtered.iterrows():
    # 如果三个情景的分类不完全相同，就认为发生了变化
    if not (row['ideal_class'] == row['baseline_class'] == row['climate_class']):
        changed_island_ids.append(row['island_id'])

print(f"在有效区域中，实际发生分类变化的岛屿数量: {len(changed_island_ids)}")

# 2. 只保留发生变化的岛屿
sankey_df = sankey_df_filtered[sankey_df_filtered['island_id'].isin(changed_island_ids)].copy()

print(f"筛选前（有效区域所有岛屿）: {len(sankey_df_filtered)} 个岛屿")
print(f"筛选后（仅变化岛屿）: {len(sankey_df)} 个岛屿")
print(f"涉及区域数: {sankey_df['region'].nunique()} 个")

# 3. 检查每个区域的变化岛屿分布
region_change_counts = sankey_df['region'].value_counts()
print(f"\n各区域的变化岛屿分布:")
for region in sorted(region_change_counts.index):
    count = region_change_counts[region]
    print(f"  {region}: {count} 个变化岛屿")

print(f"\n=== 桑基图将只显示实际发生分类变化的 {len(sankey_df)} 个岛屿 ===")


In [ ]:
# 统计每个情景下每个区域在不同分类的岛屿数量
import pandas as pd

# 准备存储结果的列表
results = []

# 对三个情景分别统计
for scenario, scenario_label in zip(['ideal_class', 'baseline_class', 'climate_class'], 
                                    ['Ideal', 'Baseline', 'Climate Stress']):
    # 按区域和分类分组统计
    counts = sankey_df.groupby(['region', scenario]).size().reset_index(name='count')
    counts['scenario'] = scenario_label
    counts = counts.rename(columns={scenario: 'classification'})
    results.append(counts)

# 合并所有结果
summary_df = pd.concat(results, ignore_index=True)

# 重新排列列顺序
summary_df = summary_df[['scenario', 'region', 'classification', 'count']]

# 排序
summary_df = summary_df.sort_values(['scenario', 'region', 'classification']).reset_index(drop=True)

print(f"=" * 100)
print(f"完整的岛屿降级分析报告")
print(f"=" * 100)

# ============================================================================
# 定义降级判断函数
# ============================================================================

# 定义六个分类的等级（数字越大越差）
classification_hierarchy = {
    "Feasible\nLow Cost\nHigh Affordability": 1,      # 最好
    "Feasible\nHigh Cost\nHigh Affordability": 2,     
    "Feasible\nLow Cost\nLow Affordability": 3,       
    "Infeasible\nHigh Cost\nHigh Affordability": 4,   
    "Infeasible\nLow Cost\nLow Affordability": 5,     
    "Infeasible\nHigh Cost\nLow Affordability": 6,    # 最差
}

def is_degradation(from_class, to_class):
    """判断是否为降级（从好变坏）"""
    return classification_hierarchy[to_class] > classification_hierarchy[from_class]

def is_upgrade(from_class, to_class):
    """判断是否为升级（从坏变好）"""
    return classification_hierarchy[to_class] < classification_hierarchy[from_class]

# ============================================================================
# 分析所有降级路径
# ============================================================================

print(f"\n{'='*100}")
print(f"第一部分：降级概览")
print(f"{'='*100}\n")

# 统计各阶段的降级
ideal_to_baseline_degrade = []
baseline_to_climate_degrade = []
continuous_degrade = []  # 连续两阶段都降级

for _, row in sankey_df.iterrows():
    ideal = row['ideal_class']
    baseline = row['baseline_class']
    climate = row['climate_class']
    region = row['region']
    island_id = row['island_id']
    
    # 检查第一阶段降级（Ideal → Baseline）
    if is_degradation(ideal, baseline):
        ideal_to_baseline_degrade.append({
            'region': region,
            'island_id': island_id,
            'from': ideal,
            'to': baseline,
            'from_level': classification_hierarchy[ideal],
            'to_level': classification_hierarchy[baseline],
            'degrade_magnitude': classification_hierarchy[baseline] - classification_hierarchy[ideal]
        })
    
    # 检查第二阶段降级（Baseline → Climate Stress）
    if is_degradation(baseline, climate):
        baseline_to_climate_degrade.append({
            'region': region,
            'island_id': island_id,
            'from': baseline,
            'to': climate,
            'from_level': classification_hierarchy[baseline],
            'to_level': classification_hierarchy[climate],
            'degrade_magnitude': classification_hierarchy[climate] - classification_hierarchy[baseline]
        })
    
    # 检查连续降级
    if is_degradation(ideal, baseline) and is_degradation(baseline, climate):
        continuous_degrade.append({
            'region': region,
            'island_id': island_id,
            'path': f"{ideal} → {baseline} → {climate}",
            'total_degrade': classification_hierarchy[climate] - classification_hierarchy[ideal]
        })

df_ideal_baseline = pd.DataFrame(ideal_to_baseline_degrade)
df_baseline_climate = pd.DataFrame(baseline_to_climate_degrade)
df_continuous = pd.DataFrame(continuous_degrade)

print(f"总变化岛屿数: {len(sankey_df)}")
print(f"第一阶段降级岛屿数 (Ideal → Baseline): {len(df_ideal_baseline)} ({len(df_ideal_baseline)/len(sankey_df)*100:.1f}%)")
print(f"第二阶段降级岛屿数 (Baseline → Climate Stress): {len(df_baseline_climate)} ({len(df_baseline_climate)/len(sankey_df)*100:.1f}%)")
print(f"连续两阶段降级岛屿数: {len(df_continuous)} ({len(df_continuous)/len(sankey_df)*100:.1f}%)")

# ============================================================================
# 按区域详细分析降级
# ============================================================================

print(f"\n{'='*100}")
print(f"第二部分：各区域降级详情")
print(f"{'='*100}\n")

all_regions = sorted(sankey_df['region'].unique())

for region in all_regions:
    region_total = len(sankey_df[sankey_df['region'] == region])
    
    # 安全地筛选各区域数据
    if len(df_ideal_baseline) > 0:
        region_ideal_baseline = df_ideal_baseline[df_ideal_baseline['region'] == region]
    else:
        region_ideal_baseline = pd.DataFrame()
    
    if len(df_baseline_climate) > 0:
        region_baseline_climate = df_baseline_climate[df_baseline_climate['region'] == region]
    else:
        region_baseline_climate = pd.DataFrame()
    
    if len(df_continuous) > 0:
        region_continuous = df_continuous[df_continuous['region'] == region]
    else:
        region_continuous = pd.DataFrame()
    
    if len(region_ideal_baseline) > 0 or len(region_baseline_climate) > 0:
        print(f"\n{'─'*100}")
        print(f"区域: {region} (共{region_total}个变化岛屿)")
        print(f"{'─'*100}")
        
        # 第一阶段降级
        if len(region_ideal_baseline) > 0:
            print(f"\n📉 第一阶段降级 (Ideal → Baseline): {len(region_ideal_baseline)}个岛屿 ({len(region_ideal_baseline)/region_total*100:.1f}%)")
            degrade_paths_1 = region_ideal_baseline.groupby(['from', 'to']).size().reset_index(name='count')
            degrade_paths_1 = degrade_paths_1.sort_values('count', ascending=False)
            for _, path in degrade_paths_1.iterrows():
                print(f"  • {path['from'].replace(chr(10), '-')} → {path['to'].replace(chr(10), '-')}: {path['count']}个")
        
        # 第二阶段降级
        if len(region_baseline_climate) > 0:
            print(f"\n📉 第二阶段降级 (Baseline → Climate Stress): {len(region_baseline_climate)}个岛屿 ({len(region_baseline_climate)/region_total*100:.1f}%)")
            degrade_paths_2 = region_baseline_climate.groupby(['from', 'to']).size().reset_index(name='count')
            degrade_paths_2 = degrade_paths_2.sort_values('count', ascending=False)
            for _, path in degrade_paths_2.iterrows():
                print(f"  • {path['from'].replace(chr(10), '-')} → {path['to'].replace(chr(10), '-')}: {path['count']}个")
        
        # 连续降级
        if len(region_continuous) > 0:
            print(f"\n⚠️  连续降级 (两阶段都恶化): {len(region_continuous)}个岛屿 ({len(region_continuous)/region_total*100:.1f}%)")
            continuous_paths = region_continuous.groupby('path').size().reset_index(name='count')
            continuous_paths = continuous_paths.sort_values('count', ascending=False)
            for _, path in continuous_paths.iterrows():
                path_str = path['path'].replace('\n', '-')
                print(f"  • {path_str}: {path['count']}个")

# ============================================================================
# 降级严重程度排名
# ============================================================================

print(f"\n{'='*100}")
print(f"第三部分：降级严重程度排名")
print(f"{'='*100}\n")

# 按区域统计降级比例
region_degrade_stats = []
if len(df_continuous) > 0:
    for region in all_regions:
        region_total = len(sankey_df[sankey_df['region'] == region])
        region_degrade = len(df_continuous[df_continuous['region'] == region])
        
        if region_degrade > 0:
            region_degrade_stats.append({
                'region': region,
                'total_islands': region_total,
                'continuous_degrade': region_degrade,
                'degrade_rate': region_degrade / region_total * 100
            })

if len(region_degrade_stats) > 0:
    df_degrade_rank = pd.DataFrame(region_degrade_stats)
    df_degrade_rank = df_degrade_rank.sort_values('degrade_rate', ascending=False)
    
    print("连续降级比例最高的区域:\n")
    for idx, row in df_degrade_rank.iterrows():
        print(f"{row['region']}: {row['continuous_degrade']}/{row['total_islands']} ({row['degrade_rate']:.1f}%) 岛屿经历连续降级")
else:
    print("没有岛屿经历连续降级")

# ============================================================================
# 汇总统计
# ============================================================================

print(f"\n{'='*100}")
print(f"第四部分：全球汇总统计")
print(f"{'='*100}\n")

total_islands = len(sankey_df)
total_ideal_baseline_degrade = len(df_ideal_baseline)
total_baseline_climate_degrade = len(df_baseline_climate)
total_continuous_degrade = len(df_continuous)

# 按降级幅度统计
print("按降级幅度分类:\n")
if len(df_continuous) > 0:
    degrade_magnitude = df_continuous.groupby('total_degrade').size().reset_index(name='count')
    degrade_magnitude['percentage'] = degrade_magnitude['count'] / total_continuous_degrade * 100
    degrade_magnitude = degrade_magnitude.sort_values('total_degrade', ascending=False)
    
    for _, row in degrade_magnitude.iterrows():
        magnitude = int(row['total_degrade'])
        count = int(row['count'])
        pct = row['percentage']
        print(f"  降级{magnitude}个等级: {count}个岛屿 ({pct:.1f}%)")
else:
    print("  没有连续降级的岛屿")

# 创建最终汇总表
print(f"\n{'='*100}")
print(f"核心数据汇总表")
print(f"{'='*100}\n")

summary_table = {
    '指标': [
        '总变化岛屿数',
        '第一阶段降级岛屿数',
        '第二阶段降级岛屿数',
        '连续两阶段降级岛屿数',
        '降级比例（第一阶段）',
        '降级比例（第二阶段）',
        '连续降级比例'
    ],
    '数值': [
        f"{total_islands}个",
        f"{total_ideal_baseline_degrade}个",
        f"{total_baseline_climate_degrade}个",
        f"{total_continuous_degrade}个",
        f"{total_ideal_baseline_degrade/total_islands*100:.1f}%",
        f"{total_baseline_climate_degrade/total_islands*100:.1f}%",
        f"{total_continuous_degrade/total_islands*100:.1f}%"
    ]
}

df_summary_table = pd.DataFrame(summary_table)
display(df_summary_table)

# ============================================================================
# 创建transition_df用于保存
# ============================================================================

# 创建变化路径：Ideal -> Baseline -> Climate Stress
sankey_df['transition_path'] = (
    sankey_df['ideal_class'] + ' → ' + 
    sankey_df['baseline_class'] + ' → ' + 
    sankey_df['climate_class']
)

# 按区域和变化路径统计
transition_stats = sankey_df.groupby(['region', 'transition_path']).size().reset_index(name='island_count')
transition_stats = transition_stats.sort_values(['region', 'island_count'], ascending=[True, False])

# 为每个区域计算百分比
transition_summary = []
for region in sorted(sankey_df['region'].unique()):
    region_data = transition_stats[transition_stats['region'] == region]
    total_islands_region = region_data['island_count'].sum()
    
    for idx, row in region_data.iterrows():
        path = row['transition_path']
        count = row['island_count']
        percentage = (count / total_islands_region * 100)
        
        transition_summary.append({
            'region': region,
            'transition_path': path,
            'island_count': count,
            'percentage': f"{percentage:.1f}%"
        })

transition_df = pd.DataFrame(transition_summary)

print(f"\n输出文件已保存:")
print(f"  - climate_transition.csv: 完整变化路径")
print(f"  - continuous_degradation_details.csv: 连续降级详细数据")
print(f"  - scenario_region_classification.csv: 各情景各区域分类统计")

# 保存所有输出文件
transition_df.to_csv('climate_transition.csv', encoding='utf-8', index=False)
if len(df_continuous) > 0:
    df_continuous.to_csv('continuous_degradation_details.csv', index=False, encoding='utf-8')
else:
    print("  注意: 没有连续降级数据，跳过保存 continuous_degradation_details.csv")
summary_df.to_csv('scenario_region_classification.csv', index=False, encoding='utf-8')

In [ ]:
# 9. --- Nature Communications风格四层桑基图：Region -> Ideal -> Baseline -> Climate Stress ---

import plotly.graph_objects as go  # 导入plotly桑基图绘制库
import plotly.io as pio  # 导入plotly IO库用于格式设置
import matplotlib.cm as cm  # matplotlib颜色映射库
import matplotlib.colors as mcolors  # matplotlib颜色工具
import numpy as np  # 数值计算库

# <<< MODIFIED: Updated print statement >>>
print("开始创建Nature Communications风格四层桑基图（使用统一的六区域颜色）...")

# 设置plotly默认样式为静态图，符合期刊要求
pio.renderers.default = "browser"  # 设置输出渲染器为浏览器

# =============================================================================
# 1. 数据检查和预处理
# =============================================================================

# 检查桑基图数据是否存在
if 'sankey_df' not in locals():
    print("错误：sankey_df数据不存在，请先运行上一个cell生成数据")
else:
    print(f"桑基图数据检查通过，包含 {len(sankey_df)} 个岛屿")
    print(f"涉及区域: {sankey_df['region'].nunique()} 个")
    print(f"涉及分类组合: {len(sankey_df[['ideal_class', 'baseline_class', 'climate_class']].drop_duplicates())} 种")

# =============================================================================
# 2. Nature风格颜色映射 - 使用统一的六区域分类颜色
# =============================================================================

# <<< MODIFIED: Define the six region names and colors >>>
# 定义六个主要类别的名称和颜色
SIX_REGION_NAMES = [
    "Feasible\nLow Cost\nHigh Affordability",
    "Feasible\nHigh Cost\nHigh Affordability",
    "Feasible\nLow Cost\nLow Affordability",
    "Infeasible\nHigh Cost\nHigh Affordability",
    "Infeasible\nLow Cost\nLow Affordability",
    "Infeasible\nHigh Cost\nLow Affordability",
]

six_region_colors_hex = {
    SIX_REGION_NAMES[0]: '#012A61',  # 深蓝 (最理想)
    SIX_REGION_NAMES[1]: '#0B75B3',  # 中蓝
    SIX_REGION_NAMES[2]: '#89CAEA',  # 浅蓝 (勉强可行)
    SIX_REGION_NAMES[3]: "#F0D2D2",  # 浅红 (成本问题)
    SIX_REGION_NAMES[4]: '#DC5654',  # 中红 (支付能力问题)
    SIX_REGION_NAMES[5]: '#982B2D',  # 深红 (最棘手)
}

# 转换为Plotly需要的rgba格式，并调整透明度
def hex_to_rgba(hex_color, alpha=0.65):
    """将十六进制颜色转换为rgba格式"""
    hex_color = hex_color.lstrip('#')
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return f'rgba({r}, {g}, {b}, {alpha})'

# 转换六区域颜色为rgba格式
six_region_colors_rgba = {name: hex_to_rgba(color, 1) for name, color in six_region_colors_hex.items()}

print(f"使用与之前图表一致的六区域颜色配色方案")


# 为区域生成高对比度的不同颜色 - 避免与分类颜色相似
def generate_distinct_region_colors(regions):
    """生成与六区域颜色高度区分的区域颜色"""
    
    # 使用一套不同的颜色系列作为区域（流）的颜色
    distinct_region_colors_1 = [
        '#60BFA4', '#ECBF51', '#760813', '#ED9D3A', '#1E7A8D', '#F06F25',
        '#5B6E82', '#F73F43', '#C7D5B2', '#C4567D', '#E4D69F', "#86484A",'#2e1e3c'
    ]
    distinct_region_colors_2 = [
        '#0E91AB', '#335D68', '#685E88', '#8AA9A0', '#835583', '#9E2A2B',
        '#C4567D', '#E19F3F', '#E29BB5', '#FCAD5F', '#AADCA2', '#E7F59A'
    ]
    distinct_region_colors_3 = [
        '#def5e5', '#b9e6c7', '#58cbad', '#40b7ad', 
        '#36a3ab', '#348fa7', '#357ba2', '#37659e', '#3e5095', 
        '#413d7b', '#3b2d5a', '#2e1e3c', '#0b0405'
    ]
    distinct_region_colors = [
        # "#240A4B",
        "#061F41",
        '#012A61',
        '#015696',
        '#0B75B3',
        '#4596CD',
        '#89CAEA',
        '#BBE6FA',
        '#FCCDC9',
        '#EE9D9F',
        '#DE6A69',
        '#C84747',
        '#982B2D',
        "#530A0B",
        # "#250303",
        ]
    region_colors = {}
    region_colors_hex = {}  # 同时保存十六进制格式用于图例
    for i, region in enumerate(regions):
        # 循环使用不同的颜色，确保区域间有明显区别
        color_idx = i % len(distinct_region_colors)
        hex_color = distinct_region_colors[color_idx]
        region_colors[region] = hex_to_rgba(hex_color, 0.55)  # 稍微透明一些以区分层级
        region_colors_hex[region] = hex_color  # 保存十六进制格式
    
    return region_colors, region_colors_hex

# 获取所有唯一区域并生成颜色
unique_regions = sorted(sankey_df['region'].unique())
region_colors, region_colors_hex = generate_distinct_region_colors(unique_regions)

print(f"为 {len(unique_regions)} 个区域生成了高对比度的区分色彩")

# =============================================================================
# 3. 构建四层桑基图的节点和链接数据结构
# =============================================================================

# 定义四个层级的所有可能节点
all_regions = sorted(sankey_df['region'].unique())  # 第1层：区域
all_ideal_classes = sorted(sankey_df['ideal_class'].unique(), key=SIX_REGION_NAMES.index)  # 第2层：Ideal分类
all_baseline_classes = sorted(sankey_df['baseline_class'].unique(), key=SIX_REGION_NAMES.index)  # 第3层：Baseline分类
all_climate_classes = sorted(sankey_df['climate_class'].unique(), key=SIX_REGION_NAMES.index)  # 第4层：Climate Stress分类


# 创建节点标签列表（按层级顺序）
node_labels = []
node_colors = []  # 节点颜色列表

# 第1层：区域节点
for region in all_regions:
    node_labels.append("")  # 使用空标签
    node_colors.append(region_colors[region])  # 使用区域对应颜色

# <<< MODIFIED: Use the new 6-region colors for the classification nodes >>>
# 第2层：Ideal分类节点
for ideal_class in all_ideal_classes:
    node_labels.append("")  # 不显示中间节点标签
    node_colors.append(six_region_colors_rgba[ideal_class])  # 使用六区域类别对应颜色

# 第3层：Baseline分类节点
for baseline_class in all_baseline_classes:
    node_labels.append("")  # 不显示中间节点标签
    node_colors.append(six_region_colors_rgba[baseline_class])

# 第4层：Climate Stress分类节点
for climate_class in all_climate_classes:
    node_labels.append("")  # 使用空标签
    node_colors.append(six_region_colors_rgba[climate_class])

print(f"创建了 {len(node_labels)} 个节点（所有标签已注释）")

# 创建节点索引映射字典
region_start_idx = 0
ideal_start_idx = len(all_regions)
baseline_start_idx = len(all_regions) + len(all_ideal_classes)
climate_start_idx = len(all_regions) + len(all_ideal_classes) + len(all_baseline_classes)

def get_node_index(layer, item):
    """获取节点索引"""
    if layer == 'region':
        return all_regions.index(item)
    elif layer == 'ideal':
        return ideal_start_idx + all_ideal_classes.index(item)
    elif layer == 'baseline':
        return baseline_start_idx + all_baseline_classes.index(item)
    else:  # climate
        return climate_start_idx + all_climate_classes.index(item)

# =============================================================================
# 4. 构建链接数据 - 保持region颜色的连续性
# =============================================================================

# 初始化链接列表
source_indices = []
target_indices = []
link_values = []
link_colors = []

print("开始构建链接关系...")

# --- 第1层到第2层的链接：Region -> Ideal ---
region_to_ideal = sankey_df.groupby(['region', 'ideal_class']).size().reset_index(name='count')
for _, row in region_to_ideal.iterrows():
    source_idx = get_node_index('region', row['region'])
    target_idx = get_node_index('ideal', row['ideal_class'])
    
    source_indices.append(source_idx)
    target_indices.append(target_idx)
    link_values.append(row['count'])
    # 使用稍微透明的区域颜色
    link_colors.append(region_colors[row['region']].replace('0.75)', '0.5)'))

print(f"第1层到第2层链接: {len(region_to_ideal)} 条")

# --- 第2层到第3层的链接：Ideal -> Baseline ---
# We need to group by region to maintain color, so we add it to the groupby
ideal_to_baseline = sankey_df.groupby(['region', 'ideal_class', 'baseline_class']).size().reset_index(name='count')
for _, row in ideal_to_baseline.iterrows():
    source_idx = get_node_index('ideal', row['ideal_class'])
    target_idx = get_node_index('baseline', row['baseline_class'])
    
    source_indices.append(source_idx)
    target_indices.append(target_idx)
    link_values.append(row['count'])
    # 使用区域颜色保持连续性
    link_colors.append(region_colors[row['region']].replace('0.75)', '0.5)'))

print(f"第2层到第3层链接: {len(ideal_to_baseline)} 条")

# --- 第3层到第4层的链接：Baseline -> Climate Stress ---
# We need to group by region to maintain color
baseline_to_climate = sankey_df.groupby(['region', 'baseline_class', 'climate_class']).size().reset_index(name='count')
for _, row in baseline_to_climate.iterrows():
    source_idx = get_node_index('baseline', row['baseline_class'])
    target_idx = get_node_index('climate', row['climate_class'])
    
    source_indices.append(source_idx)
    target_indices.append(target_idx)
    link_values.append(row['count'])
    # 使用区域颜色保持连续性
    link_colors.append(region_colors[row['region']].replace('0.75)', '0.5)'))

print(f"第3层到第4层链接: {len(baseline_to_climate)} 条")
print(f"总链接数: {len(source_indices)}")

# =============================================================================
# 5. 创建Nature风格的桑基图 - 高分辨率设置
# =============================================================================

# 创建桑基图对象 - 直接设置为高分辨率，增大节点厚度
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=40,  # 增大节点间距适应高分辨率 (原来20)
        thickness=50,  # 增大节点厚度适应高分辨率 (原来25)
        line=dict(color="rgba(200, 200, 200, 0.5)", width=1.6),  # 增粗边框适应高分辨率 (原来0.8)
        label=node_labels,  # 所有标签都已注释为空字符串
        color=node_colors,
        # 四列布局
        x=[0.05 if i < len(all_regions) else
           0.35 if i < len(all_regions) + len(all_ideal_classes) else
           0.65 if i < len(all_regions) + len(all_ideal_classes) + len(all_baseline_classes) else
           0.95
           for i in range(len(node_labels))],
        # Arrange nodes vertically within each column
        y=[i / (max(len(all_regions) - 1, 1)) if i < len(all_regions) else
           (i - len(all_regions)) / (max(len(all_ideal_classes) - 1, 1)) if i < len(all_regions) + len(all_ideal_classes) else
           (i - len(all_regions) - len(all_ideal_classes)) / (max(len(all_baseline_classes) - 1, 1)) if i < len(all_regions) + len(all_ideal_classes) + len(all_baseline_classes) else
           (i - len(all_regions) - len(all_ideal_classes) - len(all_baseline_classes)) / (max(len(all_climate_classes) - 1, 1))
           for i in range(len(node_labels))]
    ),
    link=dict(
        source=source_indices,
        target=target_indices,
        value=link_values,
        color=link_colors,
        line=dict(color="rgba(255, 255, 255, 0.3)", width=0.6)  # 增粗链接边框适应高分辨率 (原来0.3)
    )
)])

# =============================================================================
# 6. 应用Nature期刊样式设置 - 高分辨率配置
# =============================================================================
fig.update_layout(
    font=dict(
        family="Arial",  # Nature期刊标准字体
        size=24,         # 增大字体大小适应高分辨率
        color="rgb(60, 60, 60)"  # 深灰色文字
    ),
    plot_bgcolor='rgb(255, 255, 255)',   # 纯白背景
    paper_bgcolor='rgb(255, 255, 255)',  # 纯白纸张背景
    width=2400,                        # 高分辨率宽度
    height=1600,                       # 高分辨率高度
    margin=dict(l=30, r=30, t=30, b=30),  # 增大边距适应高分辨率
    showlegend=False  # 不显示图例
)

# 显示桑基图
fig.show()

# =============================================================================
# 7. 创建详细的颜色图例
# =============================================================================

print(f"\n" + "="*80)
print(f"{'桑基图颜色图例':^80}")
print(f"="*80)

# <<< MODIFIED: Print the 6-region legend >>>
# 打印六区域分类颜色图例
print(f"\n🎨 六区域分类颜色（节点颜色）:")
print(f"-" * 60)
for i, (category, hex_color) in enumerate(six_region_colors_hex.items(), 1):
    display_name = category.replace('\n', '\n      ')
    print(f"{i:2d}. {display_name}")
    print(f"      颜色代码: {hex_color}")
    print()


# 打印区域颜色图例
print(f"🌍 IPCC区域颜色（流向颜色）:")
print(f"-" * 60)
# 获取该桑基图中实际出现的区域及其岛屿数量
region_island_counts = sankey_df['region'].value_counts().sort_index()

for i, region in enumerate(sorted(unique_regions), 1):
    island_count = region_island_counts.get(region, 0)
    hex_color = region_colors_hex[region]
    print(f"{i:2d}. {region} (包含 {island_count} 个变化岛屿)")
    print(f"     颜色代码: {hex_color}")
    print()

# 打印图例说明
print(f"📋 图例说明:")
print(f"-" * 60)
print(f"• 节点颜色: 代表六区域分类，基于统一的output_0中位数基准")
print(f"• 流向颜色: 代表IPCC区域，追踪岛屿从各区域的流动路径")
print(f"• 桑基图层级: 区域 → Ideal情景 → Baseline情景 → Climate Stress情景")
print(f"• 数据范围: 仅显示在三个情景中都存在的 {len(sankey_df)} 个岛屿")
print(f"• 涉及区域: {len(unique_regions)} 个IPCC区域")
print(f"• 分辨率: 2400×1600像素高分辨率显示，节点厚度50，间距40")


# =============================================================================
# 8.1. 创建分类图例 (一行五个布局)
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# --- MODIFIED: Adjust figsize for a very wide, short layout ---
fig_class_legend, ax1 = plt.subplots(figsize=(20, 4), dpi=300)

# --- MODIFIED: Adjust axis limits for a 1x5 row ---
ax1.set_xlim(0, 26)
ax1.set_ylim(0, 4)
# ax1.set_title('Six Region Classification Colors (First 5 Shown)', fontsize=22, fontweight='bold', pad=20)

# --- MODIFIED: Define positions for a single row ---
# X positions for the five items in the row
x_rect_positions = [1, 6, 11, 16, 21]
x_text_positions = [3, 8, 13, 18, 23]
# A single Y position for the row
y_pos = 1.2

# --- MODIFIED: Loop to place the first 5 items in a row ---
# NOTE: We are slicing the list to only get the first 5 categories.
for i, category in enumerate(all_ideal_classes[:5]):
    
    # Get the x coordinate from our predefined positions
    x_rect = x_rect_positions[i]
    x_text = x_text_positions[i]
    
    hex_color = six_region_colors_hex[category]
    
    # 绘制颜色方块 at the new (x, y) position
    rect = patches.Rectangle((x_rect, y_pos), 1.5, 1.2,
                             linewidth=1.5, edgecolor='black', facecolor=hex_color)
    ax1.add_patch(rect)
    
    # 添加分类名称 at the new (x, y) position
    display_name = category
    ax1.text(x_text, y_pos + 0.6, display_name,
             fontsize=20, va='center', fontweight='normal', linespacing=1.5)

# --- 隐藏坐标轴和边框 (No change here) ---
ax1.set_xticks([])
ax1.set_yticks([])
ax1.spines[['top', 'right', 'bottom', 'left']].set_visible(False)

# --- 显示图表 (No change here) ---
plt.tight_layout()
_save_pdf()
plt.show()

# =============================================================================
# 8.2. 创建区域图例 (第二个独立图表)
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 创建第二个图例的画布
# The figsize height can be adjusted based on the number of regions
fig_region_legend, ax2 = plt.subplots(figsize=(8, 8), dpi=300)

# --- 设置坐标轴和标题 ---
ax2.set_xlim(0, 10)
ax2.set_ylim(0, len(unique_regions) + 1)
ax2.set_title('IPCC Regional Colors', fontsize=22, fontweight='bold', pad=20)

# --- 绘制图例项 ---
for i, region in enumerate(unique_regions):
    y_pos = len(unique_regions) - i
    island_count = region_island_counts.get(region, 0)
    hex_color = region_colors_hex[region]
    
    # 绘制颜色方块
    rect = patches.Rectangle((1, y_pos - 0.4), 1, 0.8, 
                             linewidth=1.5, edgecolor='black', facecolor=hex_color)
    ax2.add_patch(rect)
    
    # 添加区域名称和岛屿数量
    ax2.text(2.5, y_pos, f'{region} ({island_count})', 
             fontsize=16, va='center', fontweight='normal')
    
    # 添加颜色代码
    ax2.text(6.5, y_pos, hex_color, 
             fontsize=14, va='center', fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='#f0f0f0', alpha=0.9))

# --- 隐藏坐标轴和边框 ---
ax2.set_xticks([])
ax2.set_yticks([])
ax2.spines[['top', 'right', 'bottom', 'left']].set_visible(False)

# --- 显示图表 ---
plt.tight_layout()
_save_pdf()
plt.show()

# =============================================================================
# 9. 打印最终摘要
# =============================================================================

print(f"\n=== 高分辨率桑基图样式摘要 ===")
# <<< MODIFIED: Updated summary text >>>
print(f"✓ 六区域分类颜色:")
for category, color in six_region_colors_hex.items():
    print(f"   {category.replace(chr(10), ' ')}: {color}") # Print on one line for summary
print(f"✓ 区域颜色: {len(unique_regions)} 种高对比度色彩，避免与分类颜色冲突")
print(f"✓ 链接透明度: 0.5，保持区域颜色连续性")
print(f"✓ Nature期刊风格: Arial字体，简洁布局")
print(f"✓ 所有节点标签已注释，呈现纯色彩流动效果")
print(f"✓ 详细颜色图例已生成，包含所有颜色映射关系")
print(f"✓ 高分辨率设置: 2400×1600像素，24号字体，节点厚度50")

print(f"\n数据统计:")
print(f"总岛屿数: {len(sankey_df)}")
print(f"涉及区域数: {len(all_regions)}")
print(f"总节点数: {len(node_labels)}")
print(f"总链接数: {len(source_indices)}")

print(f"\n=== Nature风格高分辨率桑基图及图例完成 ===")
print("✓ 使用统一的六区域颜色方案")
print("✓ 区域使用高对比度色彩，避免视觉混淆")
print("✓ 纯色彩流动，更加简洁优雅")
print("✓ 期刊级别的图形质量，2400×1600高分辨率")
print("✓ 完整的颜色图例，便于理解各颜色含义")
print("✓ 节点和边框粗细已适配高分辨率显示")